# The pilot lines (chapter 11)

Work out the break even, the kill line, the noise margin, the pass line and the sample size from your own figures. The spreads come from Kestrel's synthetic fill records: lager fills on twelve 600 hL fermenters in six matched pairs.

In [ ]:
import pandas as pd
inp = pd.read_csv("../data/kestrel_inputs.csv").set_index("item")["value"]
inp

## The spreads, from the fill records (11.5)

Single fills spread about 0.6 points. The gap between the two tanks of a pair filled in the same week spreads about 0.4.

In [ ]:
fills = pd.read_csv("../data/fill_records.csv")
fills["side"] = fills["tank"].str[2:].astype(int) % 2
pairs = fills.pivot_table(index=["pair", "cycle"], columns="side", values="loss_pct")
gap = pairs[1] - pairs[0]
print(f"Fills: {len(fills)}; mean loss {fills['loss_pct'].mean():.1f} percent")
print(f"Spread of single fills {fills['loss_pct'].std():.1f} points; spread of pair gaps {gap.std():.1f} points")

## The lines

The break even uses the three year whole cost, with the ramp, at the value the planner signed.

In [ ]:
import math
from scipy import stats

value = inp["value_lost_hL_upper_bound_usd"]
whole_cost_3y = 79_200            # chapter 9, or the whole-cost notebook
ramp_years = 2.625                # 0.625 + 2, chapter 8.1
saving_per_point = inp["volume_hL_a_year"] / 100 * value
break_even = whole_cost_3y / (saving_per_point * ramp_years)
kill = math.ceil(break_even * 10) / 10
pairs_n, fills_per_tank, pair_gap_sd = 6, 2, 0.4
per_pair = pair_gap_sd / math.sqrt(fills_per_tank)
se = per_pair / math.sqrt(pairs_n)
t_value = stats.t.ppf(0.95, pairs_n - 1)
margin = 2.0 * se
print(f"Saving per point {saving_per_point:,.0f}; three year break even {break_even:.2f} points; kill line {kill}")
print(f"Steady break even at USD 30: {21_880 / (inp['volume_hL_a_year'] / 100 * 30):.2f} points")
print(f"Kill line check: {kill * 1000 * value:,.0f} a year, {kill * 1000 * value * ramp_years:,.0f} over three years")
print(f"Pair figure spread {per_pair:.2f}; standard error {se:.3f}; t for {pairs_n} pairs {t_value:.2f}")
print(f"Noise margin {margin:.2f}; pass line {kill + round(margin, 2):.2f}")

## Sample size, by Lehr's rule for paired data

About 8 x spread squared / difference squared. A data person should check it on your own records.

In [ ]:
def lehr(spread, diff):
    return 8 * spread ** 2 / diff ** 2

print(f"Full claim against zero: {lehr(0.4, 0.48):.1f} pair gaps")
print(f"Full claim against the kill line: {lehr(0.4, 0.48 - kill):.0f} pair gaps")